In [4]:
import requests

url = "https://www.uclahealth.org/sites/default/files/documents/e7/tcm-food-recommendations.pdf?f=f0952718"

response = requests.get(url)

with open("tcm_food_recommendations.pdf", "wb") as f:
    f.write(response.content)

print("Downloaded!")

Downloaded!


In [5]:
import pdfplumber
import pandas as pd
#create a empty notebook
text = ""

#+"\n": to start a new line
#copy the exact text page by page
#with automatically close the file when you finish
with pdfplumber.open("tcm_food_recommendations.pdf") as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

print(text)

TRADITIONAL CHINESE
MEDICINE FOOD
RECOMMENDATIONS
Cooling Foods
Bitter, sour, green, blue, white
Green leafy vegetables, celery, sprouts
Seeds: sesame, flax, pumpkin, hemp
Fruits: pear, watermelon, melons, pomegranate, berries, cucumber, banana
Legumes: mung bean
Fish, shellfish, and seafoods
Teas: mint, chrysanthemum, flower
Warming Foods
Sweet, spicy, red, orange
Root vegetables
Nuts
Spices: ginger, black pepper, cinnamon, turmeric, garlic
Fruits: stone fruits, cherry, dates, mango
Poultry, duck, lamb, beef
Teas: ginger, cinnamon
Moisten Dryness
Fruit: pear, berries, goji berries, melon
Legume: soybeans/tofu
Seeds: sesame, flaxseed, hemp, pumpkin
Snow fungus, wood ear
Aloe
Drinks: coconut water, honey
Resolve Dampness
Spices: nutmeg
Grains: coix seed, millet, barley, buckwheat, quinoa
Legumes: mung bean, adzuki beans
Vegetables: cabbage, bean sprouts, bamboo shoots, seaweed
Qi Tonic
Whole grains: millet, oats, brown rice
Chicken
Root vegetables: sweet potato, carrots, beets, taro, Ch

In [15]:
rows = []
# Function headings we want to recognize
functions = [
    "Cooling Foods",
    "Warming Foods",
    "Moisten Dryness",
    "Resolve Dampness",
    "Qi Tonic",
    "Blood Tonic",
    "Fire Element",
    "Earth Element",
    "Metal Element",
    "Water Element",
    "Wood Element"
]
#lines = [] is a list
#if line.strip(): only include if it wasn't empty after stripping
#splitlines(): break text into a list
#strip(): remove space from both endings of the string
lines = [line.strip() for line in text.strip().splitlines() if line.strip()]

current_function = None

print("| Food | Food Category | Function |")
print("| ---- | ---- | ---- |")


for line in lines:

    #is this the function heading?
    if line in functions:
        current_function = line
        continue


    # Ignore the title at the beginning
    if line in [
        "TRADITIONAL CHINESE",
        "MEDICINE FOOD",
        "RECOMMENDATIONS"
    ]:
        continue

    # Ignore descriptive/color lines
    if line in [
        "Bitter, sour, green, blue, white",
        "Sweet, spicy, red, orange"
    ]:
        continue

    # Category: foods
    
    if ":" in line:
        #split(":",1): use colon ":" as a cutting point
        #and stop after one "1" split
        #(example) Teas: goji berry/jujube, longan, cinnamon
        #Teas will be the category
        category, foods = line.split(":", 1)

        for food in foods.split(","):
            
            food = food.strip()
            if food:
                    rows.append({
                        "Food": food,
                        "Food Category": category.strip(),
                        "Function": current_function
                    })
            # if food:
            #     #f": formatted string
            #     #replaces the things inside {} with their values.
            #     print(
            #         f"| {food} | {category.strip()} | {current_function} |"
            #     )

    # Lines without a category
    else:
        for food in line.split(","):
            food = food.strip()
            if food:
                    rows.append({
                        "Food": food,
                        "Food Category":"General",
                        "Function": current_function
                    })

            # if food:
            #     print(
            #         f"| {food} | General | {current_function} |"
            #     )
print (rows)            

| Food | Food Category | Function |
| ---- | ---- | ---- |
[{'Food': 'Green leafy vegetables', 'Food Category': 'General', 'Function': 'Cooling Foods'}, {'Food': 'celery', 'Food Category': 'General', 'Function': 'Cooling Foods'}, {'Food': 'sprouts', 'Food Category': 'General', 'Function': 'Cooling Foods'}, {'Food': 'sesame', 'Food Category': 'Seeds', 'Function': 'Cooling Foods'}, {'Food': 'flax', 'Food Category': 'Seeds', 'Function': 'Cooling Foods'}, {'Food': 'pumpkin', 'Food Category': 'Seeds', 'Function': 'Cooling Foods'}, {'Food': 'hemp', 'Food Category': 'Seeds', 'Function': 'Cooling Foods'}, {'Food': 'pear', 'Food Category': 'Fruits', 'Function': 'Cooling Foods'}, {'Food': 'watermelon', 'Food Category': 'Fruits', 'Function': 'Cooling Foods'}, {'Food': 'melons', 'Food Category': 'Fruits', 'Function': 'Cooling Foods'}, {'Food': 'pomegranate', 'Food Category': 'Fruits', 'Function': 'Cooling Foods'}, {'Food': 'berries', 'Food Category': 'Fruits', 'Function': 'Cooling Foods'}, {'Food'

In [46]:
import pandas as pd
df = pd.DataFrame(rows)
#insert a explicit ID
# df.insert(0, "ID", ...):Put a new column at position 0, call it "ID".
# len(df):How many rows are in df= 209
#range (1,210)
#starts at 1 and stop before 210
df.insert(0, "ID", range(1, len(df) + 1))
#capitalized the first letter of Food
#pandas Series doesn't have a regular .capitalize() method.
#.str:"Apply the string operation to the strings inside this column."
#if food has "and", replace it with ""
df["Food"] = df["Food"].str.replace("and ", "", regex=False)
df["Food"] = df["Food"].str.capitalize()
df

,ID,Food,Food Category,Function
0,1,Green leafy vegetables,General,Cooling Foods
1,2,Celery,General,Cooling Foods
2,3,Sprouts,General,Cooling Foods
3,4,Sesame,Seeds,Cooling Foods
4,5,Flax,Seeds,Cooling Foods
...,...,...,...,...
204,205,Garlic,Spices,Wood Element
205,206,Green onion,Spices,Wood Element
206,207,Cilantro,Spices,Wood Element
207,208,Chrysanthemum,Teas,Wood Element


In [47]:
#data cleaning
#correction to categories
#original version: 184	Black rice	Grain/legume
#185	Black kidney bean	Grain/legume
category_corrections = {
     "Black rice": "Grains",
    "Black kidney bean": "Legumes"}
 #aplly to dataframe
#From the rows where Food is Black rice or Black kidney bean,
#select the Food Category column.
df.loc[
    df["Food"].isin(category_corrections),
    "Food Category"
 #From those same rows, give me the Food column   
] = df.loc[
    df["Food"].isin(category_corrections),
    "Food"
    #take food name (Black rice, Black kidney bean) and look them up
].map(category_corrections)
# #double check
df[df["Food"].isin(["Black rice", "Black kidney bean"])]

,ID,Food,Food Category,Function
183,184,Black rice,Grains,Water Element
184,185,Black kidney bean,Legumes,Water Element


In [48]:
#standardize my categories
df["Food Category"] = df["Food Category"].replace({
    "Beans and legumes": "Legumes",
    "Lentils": "Legumes",
    "Whole grains": "Grains",
    "Fruit":"Fruits",
    "Grain":"Grains",
    "Legume":"Legumes",
    "Tea":"Teas",
    "Spice":"Spices",
    "Liquids": "Liquid",
    "Root vegetables": "Vegetables",

})

In [49]:
df.iloc[183:197]

,ID,Food,Food Category,Function
183,184,Black rice,Grains,Water Element
184,185,Black kidney bean,Legumes,Water Element
185,186,Black sesame,Seeds,Water Element
186,187,Cinnamon,Spices,Water Element
187,188,Star anise,Spices,Water Element
188,189,Bone broth,Liquid,Water Element
189,190,Royal jelly,Liquid,Water Element
190,191,Black sesame,Teas,Water Element
191,192,Mulberry,Teas,Water Element
192,193,Shellfish,Protein,Wood Element


In [50]:
df.to_excel("TCM_food_recommendations_v8.xlsx", index=False)
import os
#getcwd(): return current working directory
print(os.getcwd())

C:\Users\rodol


In [41]:
#dropna():drop any row containing at least one missing value.
#unique():extract on one-of-a-kind item
functions = df["Function"].dropna().unique()
#enumerate():allows you to loop through an iterable (like a list, tuple, or string) 
#while automatically keeping track of the index of each item.
#(functions,1): start counting at 1, instead of 0
for i, function in enumerate(functions, 1):
    print(f"{i}. {function}")

1. Cooling Foods
2. Warming Foods
3. Moisten Dryness
4. Resolve Dampness
5. Qi Tonic
6. Blood Tonic
7. Fire Element
8. Earth Element
9. Metal Element
10. Water Element
11. Wood Element


In [43]:
choice = int(input("Enter the number of the function you're interested in: "))
#Python indexes from 0, human indexes from 1, 
#so we need to convert "choice" to Python index
selected_function = functions[choice - 1]

print(f"\nYou selected: {selected_function}")

Enter the number of the function you're interested in:  3



You selected: Moisten Dryness


In [44]:
#Take my original df, and keep only the rows where the checklist says True
#save it to a smaller df, named "recommendations"
recommendations = df[df["Function"] == selected_function]

recommendations

,ID,Food,Food Category,Function
38,39,Pear,Fruit,Moisten Dryness
39,40,Berries,Fruit,Moisten Dryness
40,41,Goji berries,Fruit,Moisten Dryness
41,42,Melon,Fruit,Moisten Dryness
42,43,Soybeans/tofu,Legume,Moisten Dryness
43,44,Sesame,Seeds,Moisten Dryness
44,45,Flaxseed,Seeds,Moisten Dryness
45,46,Hemp,Seeds,Moisten Dryness
46,47,Pumpkin,Seeds,Moisten Dryness
47,48,Snow fungus,Seeds,Moisten Dryness


In [48]:
#output prettier
#f-string: Replace anything inside {} with its value
# \n: is the newline character escape sequence, 
#used to insert a line break within a string literal *ENter, enter*
print(f"\nFoods listed under {selected_function}:\n")
#df.iterrows(): loop through the rows of a DataFrame, yielding the index and the row data as a Series for each iteration.
#_: means I don't want the index
for _, row in recommendations.iterrows():
    print(f"- {row['Food']} ({row['Food Category']})")


Foods listed under Moisten Dryness:

- Pear (Fruit)
- Berries (Fruit)
- Goji berries (Fruit)
- Melon (Fruit)
- Soybeans/tofu (Legume)
- Sesame (Seeds)
- Flaxseed (Seeds)
- Hemp (Seeds)
- Pumpkin (Seeds)
- Snow fungus (Seeds)
- Wood ear (Seeds)
- Aloe (Seeds)
- Coconut water (Drinks)
- Honey (Drinks)


In [ ]:
# What would you like to explore?

# 1. Find foods by TCM function
# 2. Find foods by food category
# 3. Find where a particular food appears
# 4. See foods that appear in multiple TCM functions
# 5. Exit

# You could eventually add things such as:

# Food
# Food Category
# TCM Function
# Protein
# Fiber
# Vitamin C
# Iron
# Calories
# ...

# Then your program could distinguish between:

# "This food is listed under Blood Tonic in the UCLA TCM resource."

# and:

# "This food also provides iron/fiber/etc."

# That distinction is important because the TCM classification and modern nutritional composition aren't the same thing.

# Current U.S. Dietary Guidelines emphasize dietary patterns containing 
# vegetables, fruits, whole grains, legumes, seafood, nuts, seeds and 
# other nutrient-dense foods rather than relying on individual "magic" foods.

In [49]:
#food search
food = input("Enter a food: ").strip().lower()

results = df[
    df["Food"].str.lower() == food
]

results

Enter a food:  pear


,ID,Food,Food Category,Function
7,8,Pear,Fruits,Cooling Foods
38,39,Pear,Fruit,Moisten Dryness
78,79,Pear,Fruits,Qi Tonic
153,154,Pear,Fruit,Metal Element
